<a id="gh200-cpu"></a>
# 01-1. GH200 — Grace CPU 컴파일과 튜닝

**세션:** 13:30–14:00  
**목표:** 같은 DGEMM 호출 코드를 두 소프트웨어 구성으로 빌드하고, 빌드 옵션이 적용되는 범위를 확인한 뒤 행렬 크기·스레드 수에 따른 성능을 측정합니다.

이 노트북은 **확인 → 수정 → 실행 → 측정 → 비교** 순서로 진행합니다. 계산 노드에서는 통합 SIF에 준비된 도구만 사용하며 패키지를 설치하거나 외부 파일을 내려받지 않습니다.


## 강의자료에서 코드로

- **Arm64 소프트웨어 생태계와 재컴파일:** 표준 인터페이스를 유지한 채 Grace에 맞게 다시 컴파일하고, 아키텍처 옵션과 NVPL 같은 최적화 라이브러리를 선택할 수 있습니다.
- **이식성과 라이브러리 추상화:** x86 전용 SIMD intrinsic을 직접 쓴 코드는 별도 이식이 필요합니다. 이번 `dgemm.c`는 BLAS 표준 인터페이스를 호출하므로 행렬 곱셈의 하드웨어 최적화는 라이브러리가 담당합니다.

따라서 이번 측정은 단순히 ‘컴파일러 하나’만 비교하지 않습니다. **GCC+OpenBLAS**와 **`nvc`+NVPL**이라는 두 소프트웨어 구성 전체를 비교합니다. 라이브러리만 공정하게 비교하려면 컴파일러와 링크 방식까지 같게 통제해야 합니다.


## 1. ARM64 GH200와 빌드 도구 확인 — 4분

`00_Start_Here`의 점검을 통과했다는 전제로 현재 계산 노드, CPU 아키텍처, GPU와 이미지 구성을 다시 확인합니다. `PASS`가 아닌 항목이 있으면 계산 노드에서 설치를 시도하지 말고 강사에게 알립니다.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import math
import os
import re
import sys

launch_dir = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in (launch_dir, *launch_dir.parents)
        if (candidate / "labs" / "gh200" / "notebook_utils.py").is_file()
    ),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("labs/gh200/notebook_utils.py를 찾지 못했습니다.")

LAB_DIR = REPO_ROOT / "labs" / "gh200"
BLAS_DIR = LAB_DIR / "blas"
WORK_DIR = REPO_ROOT / "work" / "gh200"
BIN_DIR = WORK_DIR / "bin"
PROFILE_DIR = WORK_DIR / "profiles"
BIN_DIR.mkdir(parents=True, exist_ok=True)
PROFILE_DIR.mkdir(parents=True, exist_ok=True)

if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))

from notebook_utils import print_tool_status, read_image_manifest, run, system_summary

required_tools = ("gcc", "nvc", "make")
if not print_tool_status(required_tools):
    raise EnvironmentError("필수 도구가 없습니다. 계산 노드에서 설치하지 말고 강사에게 알리세요.")

summary = system_summary()
for key, value in summary.items():
    print(f"{key:14s}: {value}")
if str(summary["architecture"]).lower() not in {"aarch64", "arm64"}:
    raise EnvironmentError(f"ARM64 환경이 아닙니다: {summary['architecture']}")
if "GH200" not in str(summary.get("gpu", "")).upper():
    raise EnvironmentError(f"GH200을 확인하지 못했습니다: {summary.get('gpu')}")

manifest = read_image_manifest()
if manifest is None:
    raise FileNotFoundError("통합 이미지 manifest를 찾지 못했습니다: /etc/ksc2026-image.json")
print("이미지 구성 정보: PASS")


## 2. 계산 코드와 빌드 명령 읽기 — 6분

BLAS(Basic Linear Algebra Subprograms)는 행렬·벡터 연산을 위한 표준 인터페이스이고, DGEMM은 배정밀도 행렬 곱셈입니다. 아래 셀은 전체 파일을 길게 출력하지 않고 계산 호출과 빌드 옵션이 있는 줄만 보여 줍니다.

| 실행 파일 | 컴파일러 | BLAS 라이브러리 | 핵심 옵션 |
|---|---|---|---|
| `dgemm-openblas` | GCC | OpenBLAS 0.3.31 | `-O3 -mcpu=native -fopenmp` |
| `dgemm-nvpl` | NVIDIA HPC SDK `nvc` | NVPL 25.5 | `-O3 -fast -mp -Mnvpl=blas` |

`-mcpu=native`는 GCC가 컴파일하는 호출 코드에 Grace CPU용 명령어와 튜닝을 선택합니다. `-fopenmp`와 `-mp`는 각 컴파일러의 OpenMP 지원을 활성화하지만, 이 소스에는 OpenMP 지시문이 없으므로 이 옵션만으로 DGEMM 호출이 병렬화되지는 않습니다. 실제 행렬 곱셈의 병렬 실행은 연결된 OpenBLAS·NVPL 구현과 실행 시 스레드 설정이 담당합니다.


In [ ]:
source_path = BLAS_DIR / "dgemm.c"
makefile_path = BLAS_DIR / "Makefile"
for path in (source_path, makefile_path):
    if not path.is_file():
        raise FileNotFoundError(path)

def show_lines(path, markers):
    print(f"\n[{path.relative_to(REPO_ROOT)}]")
    for number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        if any(marker in line for marker in markers):
            print(f"{number:3d}: {line}")

show_lines(source_path, ("extern void dgemm_", "dgemm_(&trans", "gflops", "checksum"))
show_lines(makefile_path, ("GCC_FLAGS", "NVC_FLAGS", "gcc $(", "nvc $(", "-Mnvpl"))


## 3. 고정된 두 구성으로 빌드하기 — 4분

이 실습에서는 비교 기준이 바뀌지 않도록 컴파일 옵션을 고정합니다. `GCC_FLAGS`와 `NVC_FLAGS`는 `dgemm_`을 호출하는 얇은 `dgemm.c` 래퍼의 컴파일·링크 방식을 설정합니다. 이미 빌드되어 있는 OpenBLAS와 NVPL의 DGEMM 커널 자체를 이 옵션으로 다시 컴파일하는 것은 아닙니다.

따라서 이후의 GCC+OpenBLAS와 `nvc`+NVPL 측정값은 컴파일러 하나가 아니라 두 소프트웨어 구성 전체의 결과입니다. 실제 터미널 명령은 다음 형태이며, `!`는 Jupyter 셀에서 셸 명령을 실행한다는 뜻입니다. 다음 실행 셀은 같은 명령을 구성하고 종료 코드까지 확인합니다.

```bash
!make -C labs/gh200/blas all GCC_FLAGS='-O3 -mcpu=native -fopenmp' NVC_FLAGS='-O3 -fast -mp'
```


In [ ]:
# 과정 고정값: 같은 빌드 구성을 모든 참가자가 사용합니다.
GCC_FLAGS = "-O3 -mcpu=native -fopenmp"
NVC_FLAGS = "-O3 -fast -mp"

print("GCC wrapper/link flags :", GCC_FLAGS)
print("nvc wrapper/link flags :", NVC_FLAGS)
print("Prebuilt BLAS kernels  : unchanged")


In [ ]:
make_variables = [
    f"BUILD_DIR={BIN_DIR}",
    "OPENBLAS_PREFIX=/opt/ksc2026/vendor/openblas",
    f"GCC_FLAGS={GCC_FLAGS}",
    f"NVC_FLAGS={NVC_FLAGS}",
]
run(["make", "clean", *make_variables], cwd=BLAS_DIR)
run(["make", "all", *make_variables], cwd=BLAS_DIR, timeout=600)

OPENBLAS_EXE = BIN_DIR / "dgemm-openblas"
NVPL_EXE = BIN_DIR / "dgemm-nvpl"
for executable in (OPENBLAS_EXE, NVPL_EXE):
    if not executable.is_file():
        raise FileNotFoundError(executable)
print("빌드 결과: PASS")


## 4. 행렬 크기를 선택해 첫 결과 얻기 — 4분

행렬 한 변의 길이 `N`을 두 배로 늘리면 DGEMM의 연산량은 대략 여덟 배가 됩니다. 스레드를 늘리면 병렬 계산 자원은 늘지만, 스레드 관리 비용과 메모리 계층 때문에 성능이 같은 비율로 증가하지는 않습니다.

1. `MY_MATRIX_SIZE`를 정합니다.
2. 현재 할당 안에서 정한 기준 스레드 수로 두 실행 파일을 실행합니다.
3. 체크섬이 일치하는지 먼저 확인한 뒤 실행 시간과 GFLOP/s를 비교합니다.


In [ ]:
affinity_cpus = (
    len(os.sched_getaffinity(0))
    if hasattr(os, "sched_getaffinity")
    else (os.cpu_count() or 1)
)
slurm_cpus_text = os.environ.get("SLURM_CPUS_PER_TASK", "").strip()
if slurm_cpus_text:
    try:
        slurm_cpus = int(slurm_cpus_text)
    except ValueError as exc:
        raise EnvironmentError(
            f"SLURM_CPUS_PER_TASK가 정수가 아닙니다: {slurm_cpus_text!r}"
        ) from exc
    if slurm_cpus < 1:
        raise EnvironmentError(f"SLURM_CPUS_PER_TASK가 올바르지 않습니다: {slurm_cpus}")
    AVAILABLE_CPUS = min(affinity_cpus, slurm_cpus)
else:
    AVAILABLE_CPUS = affinity_cpus

# 참가자 수정 ①: 안전 범위 안에서 행렬 크기를 바꾸세요.
MY_MATRIX_SIZE = 2048
BASELINE_THREADS = min(16, AVAILABLE_CPUS)
MY_REPEATS = 3

if not 256 <= MY_MATRIX_SIZE <= 8192:
    raise ValueError("MY_MATRIX_SIZE는 256 이상 8192 이하로 설정하세요.")
if not 1 <= BASELINE_THREADS <= AVAILABLE_CPUS:
    raise ValueError(f"BASELINE_THREADS는 1 이상 {AVAILABLE_CPUS} 이하이어야 합니다.")
if not 1 <= MY_REPEATS <= 10:
    raise ValueError("MY_REPEATS는 1 이상 10 이하로 설정하세요.")

print("배정된 CPU 범위    :", AVAILABLE_CPUS)
print("선택한 행렬 크기 N :", MY_MATRIX_SIZE)
print("기준 스레드 수     :", BASELINE_THREADS)


In [ ]:
def parse_dgemm_output(text):
    pattern = re.compile(
        r"n=(?P<n>\d+) repeats=(?P<repeats>\d+) "
        r"best_seconds=(?P<seconds>[0-9.eE+-]+) "
        r"gflops=(?P<gflops>[0-9.eE+-]+) checksum=(?P<checksum>[0-9.eE+-]+)"
    )
    match = pattern.search(text)
    if match is None:
        raise ValueError(f"DGEMM 결과를 해석할 수 없습니다: {text}")
    values = match.groupdict()
    return {
        "n": int(values["n"]),
        "repeats": int(values["repeats"]),
        "seconds": float(values["seconds"]),
        "gflops": float(values["gflops"]),
        "checksum": float(values["checksum"]),
    }

run_env = {
    "OMP_NUM_THREADS": str(BASELINE_THREADS),
    "OPENBLAS_NUM_THREADS": str(BASELINE_THREADS),
}
FIRST_RESULTS = {}
for stack, executable in (("GCC + OpenBLAS", OPENBLAS_EXE), ("nvc + NVPL", NVPL_EXE)):
    completed = run(
        [executable, str(MY_MATRIX_SIZE), str(MY_REPEATS)],
        env=run_env,
        timeout=600,
    )
    FIRST_RESULTS[stack] = parse_dgemm_output(completed.stdout)

checksums = [result["checksum"] for result in FIRST_RESULTS.values()]
CHECKSUM_READY = math.isclose(checksums[0], checksums[1], rel_tol=1e-8, abs_tol=1e-10)
print(f"\nChecksum comparison: {'PASS' if CHECKSUM_READY else 'FAIL'}")
if not CHECKSUM_READY:
    raise RuntimeError(f"두 구성의 checksum이 일치하지 않습니다: {checksums}")

print(f"{'Stack':18s} {'Seconds':>10s} {'GFLOP/s':>12s}")
for stack, result in FIRST_RESULTS.items():
    print(f"{stack:18s} {result['seconds']:10.4f} {result['gflops']:12.1f}")


## 5. 한 번에 한 변수만 바꾸기 — 8분

이번에는 고정된 컴파일 옵션과 선택한 행렬 크기를 그대로 두고 **스레드 수만** 바꿉니다. `THREAD_EXPERIMENTS`에서 비교할 값을 추가하거나 빼되, 1을 포함하고 현재 Slurm 작업에 배정된 CPU 수를 넘기지 않습니다. 실행 셀은 스레드 수별 GFLOP/s를 측정하고 다음 단계의 자동 요약에 전달합니다.


In [ ]:
# 참가자 수정 ②: 비교할 스레드 수를 고르세요. 1은 유지합니다.
THREAD_EXPERIMENTS = sorted(
    {1, min(8, AVAILABLE_CPUS), min(16, AVAILABLE_CPUS)}
)
if 1 not in THREAD_EXPERIMENTS:
    raise ValueError("THREAD_EXPERIMENTS에는 1이 포함되어야 합니다.")
if any(not 1 <= value <= AVAILABLE_CPUS for value in THREAD_EXPERIMENTS):
    raise ValueError(f"스레드 수는 1 이상 {AVAILABLE_CPUS} 이하이어야 합니다.")

TUNING_RESULTS = []
for threads in THREAD_EXPERIMENTS:
    env = {
        "OMP_NUM_THREADS": str(threads),
        "OPENBLAS_NUM_THREADS": str(threads),
    }
    for stack, executable in (("OpenBLAS", OPENBLAS_EXE), ("NVPL", NVPL_EXE)):
        completed = run(
            [executable, str(MY_MATRIX_SIZE), "2"],
            env=env,
            timeout=600,
        )
        result = parse_dgemm_output(completed.stdout)
        TUNING_RESULTS.append({"stack": stack, "threads": threads, **result})

print(f"\n{'Stack':10s} {'N':>6s} {'Threads':>8s} {'Seconds':>10s} {'GFLOP/s':>12s}")
for result in TUNING_RESULTS:
    print(
        f"{result['stack']:10s} {result['n']:6d} {result['threads']:8d} "
        f"{result['seconds']:10.4f} {result['gflops']:12.1f}"
    )


## 6. 측정 결과 자동 요약 — 4분

아래 셀은 `TUNING_RESULTS`에서 각 구성의 최고 스레드 수·GFLOP/s와 1스레드 대비 최고 성능 배수를 계산합니다. 할당된 CPU가 1개여서 비교 조건이 하나뿐이면 배수 대신 비교 불가 사유를 표시합니다. 결과와 실행 조건은 UTC 시각을 붙인 JSON 파일에 함께 저장됩니다.


In [ ]:
AUTOMATIC_SUMMARY = []
print(f"{'Stack':10s} {'Best threads':>12s} {'Best GFLOP/s':>14s} {'1→best speedup':>18s}")
for stack in ("OpenBLAS", "NVPL"):
    stack_rows = [row for row in TUNING_RESULTS if row["stack"] == stack]
    if not stack_rows:
        raise RuntimeError(f"{stack} 측정 결과가 없습니다.")

    best = max(stack_rows, key=lambda row: row["gflops"])
    measured_threads = sorted({row["threads"] for row in stack_rows})
    one_thread = next((row for row in stack_rows if row["threads"] == 1), None)
    if len(measured_threads) == 1:
        speedup = None
        speedup_text = "N/A (비교 조건 1개)"
    elif one_thread is None or one_thread["gflops"] <= 0:
        speedup = None
        speedup_text = "N/A (1-thread 없음)"
    else:
        speedup = best["gflops"] / one_thread["gflops"]
        speedup_text = f"{speedup:.2f}x"

    AUTOMATIC_SUMMARY.append(
        {
            "stack": stack,
            "best_threads": best["threads"],
            "best_gflops": best["gflops"],
            "speedup_1_to_best": speedup,
        }
    )
    print(
        f"{stack:10s} {best['threads']:12d} {best['gflops']:14.1f} {speedup_text:>18s}"
    )

created_utc = datetime.now(timezone.utc)
CPU_RESULT_PATH = WORK_DIR / f"cpu_results_{created_utc.strftime('%Y%m%dT%H%M%S_%fZ')}.json"
CPU_RESULT_PATH.write_text(
    json.dumps(
        {
            "created_utc": created_utc.isoformat(),
            "build_flags": {"gcc_openblas": GCC_FLAGS, "nvc_nvpl": NVC_FLAGS},
            "matrix_size": MY_MATRIX_SIZE,
            "baseline_threads": BASELINE_THREADS,
            "requested_repeats": MY_REPEATS,
            "first_results": FIRST_RESULTS,
            "thread_experiments": TUNING_RESULTS,
            "automatic_summary": AUTOMATIC_SUMMARY,
        },
        ensure_ascii=False,
        indent=2,
    ) + "\n",
    encoding="utf-8",
)
print(f"\n결과 저장: {CPU_RESULT_PATH.relative_to(REPO_ROOT)}")


## 완료 확인

- [ ] `dgemm-openblas`와 `dgemm-nvpl`을 고정된 구성으로 직접 빌드했습니다.
- [ ] 컴파일 옵션이 얇은 호출 래퍼와 링크에 적용되며, 미리 빌드된 BLAS 커널을 다시 컴파일하지 않는다는 점을 확인했습니다.
- [ ] 행렬 크기와 비교할 스레드 수를 직접 바꿨습니다.
- [ ] 체크섬이 일치한 뒤에 실행 시간과 GFLOP/s를 비교했습니다.
- [ ] 최고 성능 조건과 1스레드 대비 성능 배수가 포함된 JSON 결과를 저장했습니다.

시간이 남으면 `MY_MATRIX_SIZE`만 4096으로 바꾸고 4–6단계를 다시 실행합니다. 행렬 크기가 커질 때 스레드별 측정값이 어떻게 달라지는지 추가로 확인할 수 있습니다.

다음으로 [02_GPU_Memory_Profile.ipynb](02_GPU_Memory_Profile.ipynb)에서 벡터 덧셈을 서로 다른 CUDA 메모리 방식으로 실행하고 프로파일링합니다.

---

## 출처와 라이선스

이 실습은 KSC 2026을 위해 별도로 작성한 소스 코드를 사용합니다. OpenBLAS, NVIDIA HPC SDK와 NVPL에는 각 원저작물의 라이선스와 고지가 적용됩니다.
